### Vector Databases (Phase 3 — Week 3)

#### Where we are

In Phase 2 you learned:

```
Text
↓
Embedding Model
↓
Vector
↓
Similarity Search (Cosine Similarity)
↓
Top-K Results
```

So you can already turn text into vectors and compare them.

But one question is still open:

***Where do we store millions of vectors, and how do we search them fast?***

That is exactly what a **Vector Database** is for.

### Why normal SQL search isn't enough

As a backend developer, your instinct is:

***"I already have a database. Why not just store everything there?"***

Good question. Let's see where SQL stops working.

#### What SQL is great at

SQL searches by **exact values** and **ranges**.

```
SELECT *
FROM employees
WHERE email = 'abc@gmail.com';
```

```
SELECT *
FROM orders
WHERE amount > 5000;
```

```
SELECT *
FROM docs
WHERE title LIKE '%password%';
```

These all rely on the **characters matching**.

#### Where SQL breaks

User asks:

```
How do I recover my account?
```

Your document says:

```
Steps to reset your forgotten password
```

Notice:

```
"recover"  ≠  "reset"
"account"  ≠  "password"
```

A SQL `LIKE` query finds **nothing**, because no words match.

The problem:

```
SQL compares TEXT.

It has no concept of MEANING.
```

#### "Then I'll just compute cosine similarity in SQL"

You could store the vectors in a normal table and, for every search, compute cosine similarity against **every single row**.

```
Query Vector
   ↓
compare with Row 1
compare with Row 2
compare with Row 3
...
compare with Row 1,000,000
   ↓
sort by score
```

This works for a few hundred rows.

But it is a **full scan every time** — slow and expensive at scale.

A normal SQL index (B-tree) **cannot help here**, because it is built for sorting by exact value, not for finding "closest point in 1536-dimensional space".

```
SQL index  →  good for: =, <, >, BETWEEN
SQL index  →  useless for: "nearest meaning"
```

This is the gap a vector database fills.

### What is a Vector Database?

A vector database is a database **built specifically to store vectors and search them by similarity**.

Think of it as:

```
Normal DB  →  stores rows, searches by exact value

Vector DB  →  stores vectors, searches by meaning
```

#### What does it actually store?

For each item, a vector DB usually stores three things:

```
1. id        → 101
2. vector    → [0.82, 0.45, -0.11, ...]
3. payload   → original text + metadata
```

Example record:

```
id:      101
vector:  [0.82, 0.45, ...]
text:    "Employees get 20 annual leave days."
metadata:
    source: "handbook.pdf"
    page: 12
    category: "leave"
```

#### Its one core job

```
Given a query vector,
return the vectors that are closest to it.
```

That's the whole purpose.

```
Query Vector
   ↓
Vector Database
   ↓
Most similar vectors (+ their text)
```

Everything else (indexing, filtering, scaling) just makes this one job **fast and reliable**.

### The naive way (and why it doesn't scale)

The simplest possible search is called **brute force** (or "flat" search):

```
Compare the query vector
with EVERY stored vector,
one by one.
```

```
Query
  ↓
Doc 1   → 0.91
Doc 2   → 0.40
Doc 3   → 0.88
...
Doc N   → 0.12
  ↓
Sort and take the best
```

This is **100% accurate** — it always finds the true closest vectors.

So what's wrong?

#### The cost

```
100 vectors        →  fast
10,000 vectors     →  okay
1,000,000 vectors  →  slow
100,000,000        →  way too slow
```

The work grows **linearly** with the number of vectors. Every search re-scans the entire collection.

```
More documents  →  slower every search
```

To stay fast at scale, we need a smarter way to organize the vectors so we **don't** compare against all of them.

That smarter organization is called an **index**.

### Indexing

An **index** is a data structure that organizes the vectors so search can **skip most of them**.

#### Library analogy

Imagine a library with 1 million books.

```
Without a catalog:
read every book to find the one you want.

With a catalog:
jump almost directly to the right shelf.
```

A vector index is that catalog — but for "meaning" instead of titles.

```
No index   →  scan all 1,000,000 vectors

With index →  check maybe a few thousand
```

#### You don't need to memorize the algorithms

You will see names like:

```
HNSW   (a graph of vectors connected to nearby vectors)
IVF    (group vectors into buckets, search only nearby buckets)
```

For now, all you need is the **mental model**:

```
An index = a smart map of the vectors
           that lets search avoid checking everything.
```

#### The trade-off (important)

An index makes search fast, but to do that it gives up a tiny bit of accuracy:

```
Brute force  →  slow,  always exact
Index        →  fast,  "almost always" exact
```

That trade-off has a name: **Approximate Nearest Neighbor** search.

### Nearest Neighbor Search

"Nearest Neighbor" just means:

```
Find the vector(s) closest to my query vector.
```

"Closest" is measured with the similarity metrics from Phase 2 (e.g. cosine similarity).

```
Query Vector
     ↓
Which stored vectors are nearest?
     ↓
Return them
```

There are two flavors.

#### 1. Exact Nearest Neighbor (kNN)

```
Checks every vector.
Guarantees the true closest results.
Slow at large scale.
```

This is the brute-force approach from earlier.

#### 2. Approximate Nearest Neighbor (ANN)

```
Uses the index to check only a small subset.
Returns results that are "almost certainly" the closest.
Extremely fast, even with millions of vectors.
```

#### The intuition

```
Exact:   "I will read all 1,000,000 to be 100% sure."

ANN:     "I will read 5,000 smart picks and be 99% sure
          in a tiny fraction of the time."
```

For real AI apps, that 99% is almost always worth the massive speed-up.

```
This is why production vector databases use ANN.
```

So when someone says a vector DB does "nearest neighbor search," they almost always mean **ANN search powered by an index**.

### Bonus: Metadata Filtering

This is the feature that makes vector DBs feel familiar to backend developers.

Alongside each vector, you can store **metadata** and filter on it — just like a SQL `WHERE` clause.

```
Record:
    vector:   [0.82, 0.45, ...]
    text:     "Q3 revenue report summary"
    metadata:
        user_id:  42
        year:     2025
        type:     "finance"
```

Now you can combine **exact filtering** with **meaning search**:

```
Find chunks similar to this question
WHERE user_id = 42
AND   year = 2025
```

#### Why this matters

```
Semantic search  →  finds the right MEANING

Metadata filter  →  keeps it to the right DATA
```

Example: in a multi-tenant app, you must only search **this user's** documents — never another customer's. Metadata filtering enforces that.

```
SQL background:   WHERE user_id = 42
Vector DB:        same idea + "similar in meaning"
```

### Popular Vector Databases

You do **not** need to install or compare these in depth yet. Just recognize the names and their general flavor.

| Database  | Flavor | Good first impression |
| --------- | ------ | --------------------- |
| Pinecone  | Managed cloud service | No servers to run; you call an API |
| Weaviate  | Open source + cloud | Feature-rich, can self-host |
| Qdrant    | Open source (written in Rust) | Fast, easy to run locally |
| Chroma    | Lightweight, local-first | Great for learning and small projects |

#### A note for backend developers

Vector search isn't only in dedicated databases. Tools you may already know have added it:

```
Postgres   →  pgvector extension
Redis      →  vector search module
Elasticsearch / OpenSearch  →  vector fields
MongoDB    →  Atlas Vector Search
```

So the choice is often:

```
"Add vectors to the database I already use"
            vs
"Use a dedicated vector database"
```

For now, the takeaway is simply:

```
Many options exist.
They all do the same core job:
store vectors + search by similarity.
```

### The Full Picture

A vector database has two phases.

#### Phase A — Indexing (done once, ahead of time)

```
Documents
    ↓
Chunking
    ↓
Embedding Model
    ↓
Vectors
    ↓
Store in Vector DB
    ↓
Build Index  ←  the "smart map"
```

#### Phase B — Querying (done on every user request)

```
User Question
    ↓
Embedding Model
    ↓
Query Vector
    ↓
Vector DB  →  ANN Search over the Index
    ↓
Top-K Nearest Vectors
    ↓
Their original text
```

Keep these two phases separate in your mind:

```
Indexing  =  preparing the data
Querying  =  searching the data
```

### Goal — Answer this question

***Why does a vector database exist?***

Because the things we want from AI search are exactly the things a normal database can't do efficiently:

```
1. Search by MEANING, not exact words
   (embeddings + similarity, not = or LIKE)

2. Do it over MILLIONS of vectors
   (indexing, so we don't scan everything)

3. Do it FAST enough for a live request
   (Approximate Nearest Neighbor search)

4. Still filter by normal fields
   (metadata filtering, like SQL WHERE)
```

In one sentence:

***A vector database exists to store embeddings and find the most similar ones quickly, at scale — something a normal SQL database is not designed to do.***

#### Coming next (Phase 4 — RAG)

You now have every building block:

```
Embeddings  +  Similarity Search  +  Vector DB
```